# Exercises on Spark SQL/DataFrame API

This notebook contains the spark exercises using the **Spark SQL/DataFrame API**.

We start by installing pyspark (only execute if this is needed, e.g., if you are running this on Google Colab), and downloading the datasets. The exercises, which are the same as for the CORE API, follow.

### Attention!

This notebook contains some linux-specific commands to download datasets. If you are running on windows, it is advised to run this notebook through Google Colab.

### Useful documentation to do these exercises.

The PySpark Documentation is available at https://spark.apache.org/docs/latest/api/python/index.html. 

Instructions on how to install PySpark on your local PC may be found at https://spark.apache.org/docs/latest/api/python/getting_started/install.html. Note that by installing PySpark in this way, you automatically have a local copy of Spark.

The Spark SQL and Dataframe API api that we use below has the following  documentation which is a useful reference to have: https://spark.apache.org/docs/latest/sql-programming-guide.html

#### Installing PySpark

In [1]:
# This installs pyspark in the current python environment.
# By installing pyspark, we automatically also install spark.
# You **need** to run this cell when running this notebook in google colab
!pip install pyspark

#### General imports and starting Spark

In [2]:
#This is needed to start a Spark session from the notebook
#You may adjust the memory used by the driver program based on your machine's settings
import os 
os.environ['PYSPARK_SUBMIT_ARGS'] ="--conf spark.driver.memory=3g  pyspark-shell"

from pyspark.sql import SparkSession

In [3]:
import os
import sys
os.chdir(r"C:\Users\gihame\Documents\UHasselt\First Year\Advanced Topic in DS\spark-exercise-assignment")
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
# -------------------------------
# Start Spark in LOCAL mode
# -------------------------------

#The following lines are just there to allow this cell to be re-executed multiple times:
#if a spark session was already started, we stop it before starting a new one
#(there can be only one spark context per jupyter notebook)
try: 
    spark
    print("Spark application already started. Terminating existing application and starting new one")
    spark.stop()
except: 
    pass

# Create a new spark session (note, the * indicates to use all available CPU cores)
spark = SparkSession \
    .builder \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .appName("demoRDD") \
    .getOrCreate()
    
#When dealing with RDDs, we work the sparkContext object. See https://spark.apache.org/docs/latest/api/python/pyspark.html#pyspark.SparkContext
sc=spark.sparkContext

# We print the sparkcontext. This prints general information about the spark instance we have connected to. 
# In particular, the hyperlink allows us to open the spark UI (useful for seeing what is going on)
# Note: this hyperlink won't work when running this notebook in Google Colab.
sc

C:\Users\gihame\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


<SparkContext master=local[*] appName=demoRDD>

### Downloading data

The next cell downloads the data required to do the exercises

In [4]:
# Local data is already in this folder. wget/unzip/ls are Linux-only.
import os
root = r"C:\Users\gihame\Documents\UHasselt\First Year\Advanced Topic in DS\spark-exercise-assignment"
os.chdir(root)
print("Working directory:", os.getcwd())
for name in ["books", "sensors", "movielens", "github"]:
    folder = os.path.join("data", name)
    print(f"{folder}: {len(os.listdir(folder))} files")

Working directory: C:\Users\gihame\Documents\UHasselt\First Year\Advanced Topic in DS\spark-exercise-assignment
data\books: 30 files
data\sensors: 1 files
data\movielens: 5 files
data\github: 2 files


## 1. Sensor data exercises
In the file “data/sensors/sensor-sample.txt” you will find on each line, multiple fields of information, let’s call them : Date(Date), Time(Time), RoomId(Integer)-SensorId(Integer), Value1(float), Value2(float)
Using this file, use spark to compute the following queries :

1. Count the number of entries for each day.
2. Count the number of measures for each pair of RoomId-SensorId.
3. Compute the average of Value1.

In [5]:
# 1. Sensor data — DataFrame API
schema = "date DATE, time TIMESTAMP, room_sensor STRING, Value1 FLOAT, Value2 FLOAT"
sensorDF = spark.read.option("delimiter", " ").csv("data/sensors/sensor-sample.txt", schema=schema)

# 1. Number of entries for each day
sensorDF.groupBy("date").count().orderBy("date").show()

# 2. Number of measures for each RoomId-SensorId
sensorDF.groupBy("room_sensor").count().orderBy("room_sensor").show()

# 3. Average of Value1
from pyspark.sql.functions import avg
sensorDF.select(avg("Value1")).show()


+----------+-----+
|      date|count|
+----------+-----+
|2017-02-28|62103|
|2017-03-01|33423|
|2017-03-02|32403|
|2017-03-03|29727|
|2017-03-04|30225|
|2017-03-05|26019|
|2017-03-06|24315|
|2017-03-07|26625|
|2017-03-08|29343|
|2017-03-09|27288|
|2017-03-10|12483|
|2017-03-11|19059|
|2017-03-12|25089|
|2017-03-13|24783|
|2017-03-14|23418|
|2017-03-15|11901|
|2017-03-16|13869|
|2017-03-17|26922|
|2017-03-18|17427|
|2017-03-19|21999|
+----------+-----+
only showing top 20 rows


+-----------+-----+
|room_sensor|count|
+-----------+-----+
|        1-0|43047|
|        1-1|43047|
|        1-2|43047|
|        2-0|46915|
|        2-1|46915|
|        2-2|46915|
|        3-0|46634|
|        3-1|46634|
|        3-2|46634|
|        4-0|43793|
|        4-1|43793|
|        4-2|43793|
|        5-0|   35|
|        5-1|   35|
|        5-2|   35|
|        6-0|35666|
|        6-1|35666|
|        6-2|35666|
|        7-0|14910|
|        7-1|14910|
+-----------+-----+
only showing top 20 rows


+-----------------+
|      avg(Value1)|
+-----------------+
|92.82106000775526|
+-----------------+



## 2. Movielens movie data exercises

Movielens (https://movielens.org/) is a website that provides non-commercial, personalised movie recommendations. GroupLens Research has collected and made available rating data sets from the MovieLens web site for the purpose of research into making recommendation services. In this exercise, we will use one of these datasets (the movielens latest dataset, http://files.grouplens.org/datasets/movielens/ml-latest-small.zip) and compute some basic queries on it.
The dataset has already been downloaded and is available at data/movielens/movies.csv, data/movielens/ratings.csv, data/movielens/tags.csv, data/movielens/links.csv

1. Inspect the dataset's [README file](http://files.grouplens.org/datasets/movielens/ml-latest-small-README.html), in particular the section titled "Content and Use of Files" to learn the structure of these three files.
2. Compute all pairs (`movieid`, `rat`) where `movieid` is a movie id (as found in ratings.csv) and `rat` is the average rating of that movie id. (Hint: use aggregateByKey to compute first the sum of all ratings as well as the number of ratings per key).
2. Compute all pairs (`title`, `rat`) where `title` is a full movie title (as found in the movies.csv file), and `rat` is the average rating of that movie (computed over all possible ratings for that movie, as found in the ratings.csv file)
3. Compute all pairs (`title`, `tag`) where `title` is a full movie title that has an average rating of at least 3.5, and `tag` is a tag for that movie (as found in the tags.csv file)

Extra: if you want to experiment with larger datasets, download the 10m dataset (http://files.grouplens.org/datasets/movielens/ml-10m.zip, 250 Mb uncompressed) and re-do the exercises above

In [6]:
# 2. Movielens — DataFrame API
# Spark's CSV reader keeps commas that sit inside quoted titles.
ratings = spark.read.csv("data/movielens/ratings.csv") \
    .toDF("userId", "movieId", "rating", "timestamp")
movies = spark.read.csv("data/movielens/movies.csv") \
    .toDF("movieId", "title", "genres")
tags = spark.read.csv("data/movielens/tags.csv") \
    .toDF("userId", "movieId", "tag", "timestamp")

ratings = ratings.withColumn("rating", ratings["rating"].cast("double"))

# (movieId, rat)
from pyspark.sql.functions import avg, col
avg_by_id = ratings.groupBy("movieId").agg(avg("rating").alias("rat"))
avg_by_id.show(5)

# (title, rat)
title_rat = avg_by_id.join(movies, "movieId").select("title", "rat")
title_rat.show(5)

# (title, tag) for movies with average rating >= 3.5
title_tag = avg_by_id.filter(col("rat") >= 3.5) \
    .join(movies, "movieId") \
    .join(tags, "movieId") \
    .select("title", "tag")
title_tag.show(5)
print("Number of (title, tag) pairs:", title_tag.count())


+-------+------------------+
|movieId|               rat|
+-------+------------------+
|    296| 4.197068403908795|
|   1090| 3.984126984126984|
| 115713|3.9107142857142856|
|   3210|3.4761904761904763|
|  88140|          3.546875|
+-------+------------------+
only showing top 5 rows


+--------------------+------------------+
|               title|               rat|
+--------------------+------------------+
| Pulp Fiction (1994)| 4.197068403908795|
|      Platoon (1986)| 3.984126984126984|
|   Ex Machina (2015)|3.9107142857142856|
|Fast Times at Rid...|3.4761904761904763|
|Captain America: ...|          3.546875|
+--------------------+------------------+
only showing top 5 rows


+-------------------+----------+
|              title|       tag|
+-------------------+----------+
|Pulp Fiction (1994)|     witty|
|Pulp Fiction (1994)|   violent|
|Pulp Fiction (1994)|  violence|
|Pulp Fiction (1994)|very funny|
|Pulp Fiction (1994)|   unusual|
+-------------------+----------+
only showing top 5 rows


Number of (title, tag) pairs: 2813


## 3. Github log data exercises
Github makes activity logs publicly available at https://www.githubarchive.org/. One such log file, which contains activity data for 2015-03-01 between 0h-1h at night, has been downloaded and is available at `data/github/2015-03-01-0.json.gz`. This (compressed) file contains multiple JSON objects, one per line. Here is a sample line of this file, neatly formatted:

`{ "id": "2614896652",
    "type": "CreateEvent",
    "actor": {
        "id": 739622,
        "login": "treydock",
        "gravatar_id": "",
        "url": "https://api.githb.com/users/treydock",
        "avatar_url": "https://avatars.githubusercontent.com/u/739622?"
    },
    "repo": {
        "id": 23934080,
        "name": "Early-Modern-OCR/emop-dashboard",
    "url": "https://api.github.com/repos/Early-Modern-OCR/emop-dashboard"
    },
    "payload": {
        "ref": "development",
        "ref_type": "branch",
        "master-branch": "master",
        "description": "",
        "pusher_type": "user",
    },
    "public": true,
    "created_at": "2015-03-01T00:00:00Z",
    "org": {
        "id": 10965476,
        "login": "Early-Modern-OCR",
        "gravatar_id": "",
        "url": "https://api.github.com/orgs/Early-Modern-OCR",
        "avatar_url": "https://avatars.githubusercontent.com/u/10965476?"
    }
}`

This log entry has `CreateEvent` type and its `payload.ref_type` is `branch` . So someone named "treydock" (`actor.login`) created a repository branch called "development" (`payload.ref`) in the first second of March 1, 2015 (`created_at`) .

1. Load the json file as spark dataframe 

2. Filter this dataframe to retain only those rows that represent push activities (where `type` equals `PushEvent`)

3. Count the number of push events.

4. Compute the number of push events, grouped per `actor.login`. 

5. Retrieve the results of (4) in sorted order, where logins with higher number of pushes come first. Retrieve the 10 first such results (which contain the highest number of pushes)

6. You are representing a company and need to retrieve the number of pushes for every employee in the company. The file `data/github/employees.txt` contains a list of all employee login names at your company.

Extra: if you want to experiment with larger datasets, download more log data from the github archive website and re-do the exercises above

In [7]:
# 3. Github logs — DataFrame API
from pyspark.sql.functions import col, desc

gitDF = spark.read.json("data/github/2015-03-01-0.json.gz")
pushes = gitDF.filter(col("type") == "PushEvent")

print("Number of push events:", pushes.count())

# number of push events per actor.login
pushes_per_login = pushes.groupBy(col("actor.login").alias("login")).count()

print("Top 10 logins by number of pushes:")
pushes_per_login.orderBy(desc("count")).show(10)

# pushes for every employee (0 when that login has no PushEvent in this file)
employees = spark.read.text("data/github/employees.txt") \
    .withColumnRenamed("value", "login") \
    .filter(col("login") != "")

employee_pushes = employees.join(pushes_per_login, "login", "left") \
    .fillna(0, subset=["count"]) \
    .orderBy("login")
employee_pushes.show(15)
print("Employees:", employee_pushes.count())


Number of push events: 8793
Top 10 logins by number of pushes:


+------------------+-----+
|             login|count|
+------------------+-----+
|      greatfirebot|  192|
|diversify-exp-user|  146|
|     KenanSulayman|   72|
|        manuelrp07|   45|
|    mirror-updates|   42|
|     tryton-mirror|   37|
|           Somasis|   26|
|   direwolf-github|   24|
|   EmanueleMinotto|   22|
|           hansliu|   21|
+------------------+-----+
only showing top 10 rows


+---------------+-----+
|          login|count|
+---------------+-----+
|       AiMadobe|    3|
|         Akkyie|    4|
|       BatMiles|   10|
|     Battleroid|    2|
|     BhawanVirk|    4|
|        BitKiwi|   12|
|    ChroniXEcho|    1|
|          DomT4|    4|
|EmanueleMinotto|   22|
|         Gix075|    1|
|   GroovyCarrot|    3|
|      Halexsson|    1|
|       IrinaDmt|    1|
|        IsaacAU|   10|
|    JustScience|    1|
+---------------+-----+
only showing top 15 rows
Employees: 207
